# prompts

> The instruction text of the tutor. You can change it on the Settings page.

Each prompt of the tutor has two parts. The variable parts (the concept, the thread, the JSON shape) stay in code. The instruction text is in this registry.
Each prompt has a name, a plain label, a note on what it changes, and its default text from code. The Settings page shows each prompt, and you can change its text there.
The tutor reads each prompt when it calls the model. Thus a change works on the next question, with no restart.

In [ ]:
#| default_exp prompts

In [ ]:
#| hide
import os, tempfile
os.environ.setdefault('LITESEARCH_CACHE', tempfile.mkdtemp(prefix='ls-'))   # and litesearch does not wait on the shared lock of another run
from fastcore.test import *
os.environ['ABHYASA_CONFIG'] = tempfile.mkdtemp()   # the settings go to a temporary folder, not to your real settings

Import the tools that this module uses. The settings come from `settings` only when a prompt is read, so the import stays light.

In [ ]:
#| export
from fastcore.utils import *

## The registry

`register` adds a prompt to `PROMPTS` and returns its default text. A `check` can refuse a bad text: it raises `ValueError` with a plain message.

In [ ]:
#| export
PROMPTS = {}   # name -> the prompt: name, label, note, default text and check

def register(name:str, label:str, note:str, dflt:str, check=None) -> str:
    "Add the prompt `name` with a plain `label`, a `note` on what it changes, its `dflt` text and an optional `check`. Return `dflt`."
    PROMPTS[name] = AttrDict(name=name, label=label, note=note, dflt=dflt, check=check)
    return dflt

`lines` gives the lines of a text that are not empty. `need_lines` makes a check that refuses a text without exactly `n` such lines.

In [ ]:
#| export
def lines(text:str) -> list:
    "The lines of `text` that are not empty, without spaces at the ends."
    return [l.strip() for l in (text or '').splitlines() if l.strip()]

def need_lines(n:int, what:str):
    "A check that refuses a text that does not have `n` lines that are not empty."
    def check(text):
        if (k := len(lines(text))) != n: raise ValueError(f'{what} needs {n} lines, one on each line. This text has {k}.')
    return check

## Read and save

`prompt` gives the saved text of a prompt, else its default. `set_prompt` saves a text in the settings, under the `prompts` key. An empty text, or a text that is the same as the default, removes the saved text. Then the default is in use again, also when a new version changes the default.

In [ ]:
#| export
def saved() -> dict:
    "The saved prompt texts: name -> text."
    from abhyasa.settings import settings
    return settings().get('prompts') or {}

def prompt(name:str) -> str:
    "The text of the prompt `name` now: the saved text, else the default."
    t = saved().get(name)
    return t if isinstance(t, str) and t.strip() else PROMPTS[name].dflt

def set_prompt(name:str, text:str='') -> str:
    "Save the text of the prompt `name`. An empty text uses the default. A text that fails the check raises `ValueError`. Return the text in use."
    from abhyasa.settings import settings, save_settings
    p, text = PROMPTS[name], (text or '').replace('\r\n', '\n').strip()
    if text and p.check: p.check(text)
    d = settings()
    ps = d.setdefault('prompts', {})
    if text and text != p.dflt.strip(): ps[name] = text
    else: ps.pop(name, None)
    save_settings(d)
    return prompt(name)

## Tests

A small prompt with a check of 2 lines. Its default is in use at first.

In [ ]:
register('greet', 'Greeting', 'The first line.', 'Say hello.\nSay your name.', need_lines(2, 'The greeting'))
test_eq(prompt('greet'), 'Say hello.\nSay your name.')

A saved text is in use at once, and the settings keep it under `prompts`.

In [ ]:
test_eq(set_prompt('greet', 'Say hi.\r\nSay your job.\n'), 'Say hi.\nSay your job.')
test_eq((prompt('greet'), saved()), ('Say hi.\nSay your job.', {'greet': 'Say hi.\nSay your job.'}))

A text that fails the check is refused with a plain message, and the saved text stays.

In [ ]:
test_fail(lambda: set_prompt('greet', 'Only one line.'), contains='needs 2 lines')
test_eq(prompt('greet'), 'Say hi.\nSay your job.')

An empty text, or only spaces, removes the saved text, so the default is in use again. A text that is the same as the default is not kept.

In [ ]:
test_eq(set_prompt('greet', '  \n '), 'Say hello.\nSay your name.')
set_prompt('greet', 'Say hello.\nSay your name.')
test_eq(saved(), {})

An unknown name is an error. Then the test prompt goes away.

In [ ]:
test_fail(lambda: prompt('nope'))
del PROMPTS['greet']

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()